In [1]:
!pip install transformers   #ট্রান্সফর্মার্স লাইব্রেরি

In [2]:
!pip install datasets #ডেটাসেটস লাইব্রেরি

In [3]:
!pip install tokenizers #টোকেনাইজার্স লাইব্রেরি

In [4]:
from datasets import load_dataset
dataset = load_dataset("imdb")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [5]:
from transformers import AutoModel,AutoTokenizer

#downlode the model and tokenizer
model_name ="bert-base-uncased"
model=AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

#user the model and tokenizer
inputs = tokenizer("Hello, Hugging Face!",return_tensors="pt")
outputs = model(**inputs)
print(outputs.last_hidden_state.shape) #example output shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 7, 768])


In [6]:
#ব্যাচ সাইজ,সিকোয়েন্স লেংথ,ডাইমেনশনাল ভেক্টর --> torch.Size([1, 7, 768])

In [7]:
##sentiment analysis
!pip install torch

from transformers import pipeline
analyzer = pipeline("sentiment-analysis", model = "distilbert-base-uncased-finetuned-sst-2-english")

texts=[
    "I LOVE PLAYING AND WATCHING CRICKET",
    "I HATE WHEN VIRAT KOHLI MISSES A CENTURY"
]
RESULTS =analyzer(texts)
for i in RESULTS:
  print(i)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'label': 'POSITIVE', 'score': 0.9994063377380371}
{'label': 'NEGATIVE', 'score': 0.9990317821502686}


In [8]:
#text Classifistion

spam_classification = pipeline("text-classification", model = "philschmid/distilbert-base-multilingual-cased-sentiment")

texts = [
    "Congratulations! You've won a 500 INR Amazon gift card. Click here to claim now.",
    "Hi Amit, let's have a meeting tomorrow at 12 PM.",
    "URGENT: Your gmail account has been compromised. Click here to secure it."
]

results = spam_classification(texts)

label_mapping = {
    'negative': 'SPAM',
    'neutral': 'NOT SPAM',
    'positive': 'NOT SPAM'
}

for result in results:
    label = label_mapping[result['label']]  # Map the label
    score = result['score']
    print(f"Label: {label}, Confidence: {score:.4f}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Label: SPAM, Confidence: 0.9927
Label: NOT SPAM, Confidence: 0.5300
Label: SPAM, Confidence: 0.7050


In [9]:
#text summaraization
!pip install transformers torch

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

#load a pre-trained model and tokenizer
model_name = "facebook/bart-large-cnn" #Example model for summarization
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Input text
text = """Virat Kohli is an Indian international cricketer known for his exceptional batting skills.He has played for the Indian national team in all formats and is a former captain.
Kohli is often referred to as 'King Kohli' and 'Chase Master' due to his remarkable ability to chase targets."""

# Tokenize input
inputs = tokenizer.encode(
    "summarize: " + text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# Generate summary
summary_ids = model.generate(
    inputs,
    max_length=50,
    min_length=25,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

# Decode summary
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Virat Kohli is an Indian international cricketer known for his exceptional batting skills. He has played for the Indian national team in all formats and is a former captain. He is often referred to as 'King Kohli'


In [10]:
#Text to Text(Translation)
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the pre-trained T5 model and tokenizer
model_name = "t5-small"   # you can also try "t5-base", "t5-large"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Input text
input_text = "Translate English to Spanish: My name is Amit Diwan, and I love cricket."

# Tokenize input
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# Generate translation
outputs = model.generate(input_ids, max_length=50)

# Decode output
translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Translated Text:", translated_text)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Translated Text: Ich spreche von Amit Diwan und ich liebe den Cricket.


In [11]:
#Qustioning answer
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
import torch

# Load model and tokenizer
model_name = "bert-large-uncased-whole-word-masking-finetuned-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# Context
context = """I am Amit Diwan from Delhi. My interests include programming and cricket.
I have created courses not only in programming, but web development and database technologies as well.
Artificial Intelligence (AI) is transforming the world by enabling machines to learn, adapt, and perform tasks."""

# Question
question = "Where is Amit Diwan based?"

# Tokenize input
inputs = tokenizer(question, context, return_tensors="pt")

# Get model outputs
outputs = model(**inputs)

# Extract scores
start_scores = outputs.start_logits
end_scores = outputs.end_logits

# Get most likely answer positions
start_index = torch.argmax(start_scores)
end_index = torch.argmax(end_scores)

# Convert tokens to answer
answer_tokens = inputs["input_ids"][0][start_index : end_index + 1]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)

print(f"Answer: {answer}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking-finetuned-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer: delhi


In [12]:
#text to image
!pip install diffusers transformer accelerate torch

from diffusers import StableDiffusionPipeline
import torch

# Load the Stable Diffusion model
model_id = "CompVis/stable-diffusion-v1-4"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipe.to(device)

# Define your text prompt
prompt = "Flying cars soar over a futuristic cityscape at sunset."

# Generate the image
with torch.autocast("cuda"):
    image = pipe(prompt).images[0]

# Save the image
image.save("generated_image.png")

print("Image saved as generated_image.png")

ERROR: Could not find a version that satisfies the requirement transformer (from versions: none)
ERROR: No matching distribution found for transformer


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/50 [00:00<?, ?it/s]

Image saved as generated_image.png


In [13]:
# Text to Video
from diffusers import StableDiffusionPipeline
import torch
import cv2
import numpy as np

# Load the model
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)

pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")

# Prompt
prompt = "A futuristic cityscape at night with flying cars, cinematic, ultra realistic"

# Generate frames
frames = []
for i in range(10):  # Generate 10 frames
    image = pipe(prompt).images[0]
    frames.append(image)

# Save frames as images
for i, frame in enumerate(frames):
    cv2.imwrite(f"frame_{i}.png", cv2.cvtColor(np.array(frame), cv2.COLOR_RGB2BGR))

# Create video
frame_rate = 5
height, width, _ = np.array(frames[0]).shape

out = cv2.VideoWriter(
    "output_video.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    frame_rate,
    (width, height)
)

# Write frames to video
for i in range(len(frames)):
    frame = cv2.imread(f"frame_{i}.png")
    out.write(frame)

out.release()

print("Video saved as output_video.mp4")

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

Video saved as output_video.mp4
